# 🎯 Açıklanabilir Aday Değerlendirme ve Sıralama Sistemi 

Bu notebook, iş ilanlarına göre adayları değerlendiren, sıralayan ve her kararın arkasındaki nedenleri **SHAP ile derinlemesine** açıklayan kapsamlı bir AI sistemi içerir.

## 
- ✅ **Her aday için bireysel SHAP waterfall chart** (neden bu skoru aldığı)
- ✅ **SHAP force plot + özet grafik** (tüm adaylar için küresel XAI)
- ✅ **Model ezberlemesin** → Early stopping + Regularization + Cross-validation
- ✅ **İyileştirilmiş Gradio UX** → Tabs, SHAP paneli, aday detay kartı
- ✅ **Açıklama metni** → Her karar için insan okuyabilir gerekçe

**Veri Seti Yolu:** `/kaggle/input/datasets/mdtalhask/ai-powered-resume-screening-dataset-2025/AI_Resume_Screening.csv`

## 📦 1. Kütüphanelerin Kurulumu

In [33]:
!pip install -q xgboost lightgbm shap gradio scikit-learn pandas numpy matplotlib seaborn plotly

In [34]:
# ============================================================
# KÜTÜPHANE İMPORTLARI
# ============================================================
# Veri işleme ve sayısal hesaplama
import pandas as pd
import numpy as np

# Matplotlib — headless (ekransız) ortam için Agg backend zorunlu
# (Kaggle/Colab gibi sunucu ortamlarında grafik penceresi açılmaz)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# İstatistiksel görselleştirme
import seaborn as sns

# Etkileşimli grafikler (Plotly)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Gereksiz uyarı mesajlarını bastır
import warnings
warnings.filterwarnings('ignore')

# ── Makine Öğrenmesi (sklearn) ──────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Güçlendirme (boosting) tabanlı modeller
import xgboost as xgb   # Gradient boosting — regularizasyon desteği güçlü
import lightgbm as lgb  # Yaprak-bazlı büyüme → daha hızlı, daha az bellek

# ── XAI: Açıklanabilir Yapay Zeka ──────────────────────────
# SHAP (SHapley Additive exPlanations):
#   Oyun teorisinden türetilmiş; her özelliğin model kararına
#   katkısını bireysel olarak hesaplar.
import shap

# ── NLP: Doğal Dil İşleme ──────────────────────────────────
# TF-IDF: İş ilanı metni ile aday becerilerini vektöre çevirir.
#   "Term Frequency–Inverse Document Frequency" — nadir kelimelere
#   daha yüksek ağırlık verir, sık geçen genel kelimeleri bastırır.
# cosine_similarity: İki vektör arasındaki açıyı ölçer (0=tamamen farklı, 1=aynı)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Yardımcı araçlar
from collections import Counter  # Beceri sayma
import re                         # Düzenli ifadeler (metin temizleme)
import io
import base64

# Gradio — web tabanlı etkileşimli arayüz
import gradio as gr

# Görsel tema ayarları
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Tüm kütüphaneler başarıyla yüklendi!")


✅ Tüm kütüphaneler başarıyla yüklendi!


## 📊 2. Veri Setinin Yüklenmesi

In [35]:
DATA_PATH = '/kaggle/input/datasets/mdtalhask/ai-powered-resume-screening-dataset-2025/AI_Resume_Screening.csv'
df = pd.read_csv(DATA_PATH)

print("=" * 80)
print("📋 VERİ SETİ ÖZETİ")
print("=" * 80)
print(f"Toplam Aday Sayısı: {len(df):,}")
print(f"Özellik Sayısı: {len(df.columns)}")
display(df.head())
display(df.describe())

📋 VERİ SETİ ÖZETİ
Toplam Aday Sayısı: 1,000
Özellik Sayısı: 11


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


,Resume_ID,Experience (Years),Salary Expectation ($),Projects Count,AI Score (0-100)
count,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000
mean,500.500000,4.896000,79994.486000,5.13300,83.950000
std,288.819436,3.112695,23048.472549,3.23137,20.983036
min,1.000000,0.000000,40085.000000,0.00000,15.000000
25%,250.750000,2.000000,60415.750000,2.00000,70.000000
50%,500.500000,5.000000,79834.500000,5.00000,100.000000
75%,750.250000,8.000000,99583.250000,8.00000,100.000000
max,1000.000000,10.000000,119901.000000,10.00000,100.000000


## 🔧 3. İş Rolü Bazlı Beceri Analizi

In [36]:
# ============================================================
# İŞ ROLÜ BAZLI BECERİ ANALİZİ
# ============================================================
# Amaç: Her iş rolü için hangi becerilerin ne kadar yaygın olduğunu
# tespit etmek. Bu bilgi daha sonra:
#   1) İlan-aday eşleştirmesinde referans olarak kullanılır.
#   2) Gradio arayüzünde "önerilen beceriler" bölümünü doldurur.

def extract_skills_for_role(df, job_role):
    """
    Belirli bir iş rolü için en yaygın becerileri çıkarır.

    Parametreler
    -----------
    df       : Ham veri seti (Skills sütunu virgülle ayrılmış beceri listesi içerir)
    job_role : Filtrelenecek iş rolü adı (ör. 'Data Scientist')

    Döndürür
    --------
    Counter : {beceri_adı: kaç adayda geçtiği} şeklinde sözlük
    """
    # Yalnızca seçili role ait satırları al
    role_df = df[df['Job Role'] == job_role]

    all_skills = []
    for skills_str in role_df['Skills']:
        # Virgülle ayrılmış beceri dizesini listeye çevir, boşlukları temizle
        skills = [s.strip() for s in skills_str.split(',')]
        all_skills.extend(skills)

    # Her becerinin kaç kez geçtiğini say
    skill_counts = Counter(all_skills)
    return skill_counts

# ── Her rol için en yaygın 10 beceriyi çıkar ──
job_roles = df['Job Role'].unique()
role_skills_map = {}

for role in job_roles:
    skill_counts = extract_skills_for_role(df, role)
    # Sadece isim listesi yeterli; sayıları saklamıyoruz
    top_skills = [skill for skill, count in skill_counts.most_common(10)]
    role_skills_map[role] = top_skills

print("📊 İş Rollerine Göre En Yaygın Beceriler:\n")
for role, skills in role_skills_map.items():
    print(f"\n{role}:")
    print(f"  Top Beceriler: {', '.join(skills[:5])}")

# ── Veri setindeki tüm benzersiz becerileri derle ──
# Bu liste daha sonra aday formu otomatik tamamlama vb. için kullanılabilir.
all_unique_skills = set()
for skills_str in df['Skills']:
    skills = [s.strip() for s in skills_str.split(',')]
    all_unique_skills.update(skills)

all_unique_skills = sorted(list(all_unique_skills))
print(f"\n\n📋 Veri Setindeki Toplam Benzersiz Beceri Sayısı: {len(all_unique_skills)}")


📊 İş Rollerine Göre En Yaygın Beceriler:


AI Researcher:
  Top Beceriler: TensorFlow, NLP, Python, Pytorch

Data Scientist:
  Top Beceriler: Machine Learning, Python, SQL, Deep Learning

Cybersecurity Analyst:
  Top Beceriler: Ethical Hacking, Linux, Cybersecurity, Networking

Software Engineer:
  Top Beceriler: Java, SQL, C++, React


📋 Veri Setindeki Toplam Benzersiz Beceri Sayısı: 14


## 🔧 4. Özellik Mühendisliği (AI Score HARİÇ)

In [37]:
# ============================================================
# ÖZELLİK MÜHENDİSLİĞİ (AI Score HARİÇ)
# ============================================================
# Neden AI Score kullanmıyoruz?
#   Veri setindeki 'AI Score' sütunu zaten modelin tahmin etmeye çalıştığı
#   sonuca sızdırılmış bilgi içerebilir (data leakage riski).
#   Bu nedenle modeli ham özelliklerle eğitiyoruz.

df_processed = df.copy()

# ── Teknoloji varlık özellikleri (0/1 ikili) ─────────────────
# str.contains() → case=False büyük/küçük harf farkını görmezden gelir
# na=False → NaN değerlerinde hata vermez, False döner
df_processed['Skills_Count'] = df_processed['Skills'].apply(lambda x: len(x.split(',')))
df_processed['Has_Python']   = df_processed['Skills'].str.contains('Python', case=False, na=False).astype(int)

# ML/AI için birden fazla anahtar kelimeyi boru (|) ile birleştiriyoruz
df_processed['Has_ML']       = df_processed['Skills'].str.contains(
    'Machine Learning|Deep Learning|TensorFlow|Pytorch', case=False, na=False).astype(int)
df_processed['Has_SQL']      = df_processed['Skills'].str.contains('SQL', case=False, na=False).astype(int)
df_processed['Has_Cloud']    = df_processed['Skills'].str.contains(
    'AWS|Azure|GCP|Cloud', case=False, na=False).astype(int)
df_processed['Has_Java']     = df_processed['Skills'].str.contains('Java', case=False, na=False).astype(int)
df_processed['Has_Security'] = df_processed['Skills'].str.contains(
    'Security|Ethical Hacking|Cybersecurity', case=False, na=False).astype(int)

# ── Sertifikasyon özellikleri ─────────────────────────────────
# NaN → sertifika yok olarak yorumlanır
df_processed['Has_Certification'] = (~df_processed['Certifications'].isna()).astype(int)
df_processed['Cert_Count']        = df_processed['Certifications'].fillna('None').apply(
    lambda x: 0 if x == 'None' else len(x.split(',')))

# ── Eğitim seviyesi ordinal kodlaması ────────────────────────
# Sıralı (ordinal) ölçek: lisans < yüksek lisans < doktora
# Model bu sayısal sırayı anlayabilir; one-hot encoding gerekmiyor.
education_scores = {'B.Sc': 1, 'B.Tech': 1, 'MBA': 2, 'M.Tech': 2, 'PhD': 3}
df_processed['Education_Score'] = df_processed['Education'].map(education_scores)

# ── Türetilmiş özellik: Proje Verimliliği ────────────────────
# Deneyim yılı başına kaç proje üretildi?
# +1 → sıfıra bölme hatasını önler (sıfır deneyimli adaylar için)
df_processed['Experience_Project_Ratio'] = df_processed.apply(
    lambda x: x['Projects Count'] / (x['Experience (Years)'] + 1), axis=1)

# ── Hedef değişken ───────────────────────────────────────────
# 'Hire' → 1 (pozitif sınıf), 'Reject' → 0 (negatif sınıf)
df_processed['Target'] = (df_processed['Recruiter Decision'] == 'Hire').astype(int)

print("✅ Özellik mühendisliği tamamlandı!")
print(f"⚠️ AI Score özellik olarak KULLANILMIYOR (overfitting önlemi)")
display(df_processed[[
    'Name', 'Skills_Count', 'Has_Python', 'Has_ML', 'Education_Score',
    'Experience_Project_Ratio', 'Target'
]].head(10))


✅ Özellik mühendisliği tamamlandı!
⚠️ AI Score özellik olarak KULLANILMIYOR (overfitting önlemi)


,Name,Skills_Count,Has_Python,Has_ML,Education_Score,Experience_Project_Ratio,Target
0,Ashley Ali,3,0,1,1,0.727273,1
1,Wesley Roman,4,1,1,2,0.090909,1
2,Corey Sanchez,3,0,0,2,3.500000,1
3,Elizabeth Carney,3,1,1,1,0.000000,1
4,Julie Hill,3,0,0,3,1.800000,1
5,Samantha Santos,4,0,0,1,0.454545,1
6,Tony Smith,3,0,0,2,1.800000,1
7,Anthony Harrison,3,0,1,2,1.750000,1
8,Nancy Jenkins,2,0,0,2,0.375000,1
9,Courtney Gibson,4,1,1,2,1.000000,0


## 🤖 5. Model Eğitimi — Ezberleme Önleme (Regularization + Early Stopping)

Model ezberlemesini önlemek için:
- **XGBoost**: `min_child_weight`, `gamma`, `reg_alpha`, `reg_lambda`, `early_stopping_rounds`
- **LightGBM**: `min_data_in_leaf`, `lambda_l1`, `lambda_l2`, `early_stopping`
- **Random Forest**: `max_features`, `min_samples_leaf`, `min_samples_split`
- **5-Fold Stratified Cross-Validation** ile gerçek genelleme ölçümü

In [38]:
feature_cols = [
    'Experience (Years)', 'Projects Count', 'Skills_Count',
    'Has_Python', 'Has_ML', 'Has_SQL', 'Has_Cloud', 'Has_Java', 'Has_Security',
    'Has_Certification', 'Cert_Count', 'Education_Score', 'Experience_Project_Ratio'
]

# Feature names for SHAP (Türkçe etiketler)
feature_names_tr = {
    'Experience (Years)': 'Deneyim (Yıl)',
    'Projects Count': 'Proje Sayısı',
    'Skills_Count': 'Beceri Sayısı',
    'Has_Python': 'Python Biliyor',
    'Has_ML': 'ML/AI Biliyor',
    'Has_SQL': 'SQL Biliyor',
    'Has_Cloud': 'Cloud (AWS/GCP/Azure)',
    'Has_Java': 'Java Biliyor',
    'Has_Security': 'Siber Güvenlik',
    'Has_Certification': 'Sertifikası Var',
    'Cert_Count': 'Sertifika Sayısı',
    'Education_Score': 'Eğitim Seviyesi',
    'Experience_Project_Ratio': 'Proje/Deneyim Oranı'
}
feature_names_list = [feature_names_tr[f] for f in feature_cols]

X = df_processed[feature_cols]
y = df_processed['Target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Validation set for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("📊 Veri Seti:")
print(f"  - Eğitim: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")
print(f"  - Özellik Sayısı: {len(feature_cols)}")
print(f"  - Hire/Reject Oranı: {y.sum()}/{len(y)-y.sum()}")

📊 Veri Seti:
  - Eğitim: 800 | Validation: 120 | Test: 200
  - Özellik Sayısı: 13
  - Hire/Reject Oranı: 812/188


In [39]:
# ============================================================
# MODEL EĞİTİMİ — EZBERLEMEYİ ÖNLEME STRATEJİLERİ
# ============================================================
#
# Overfitting (ezberleme) nedir?
#   Modelin eğitim verisini ezberleyip yeni (test) veriye
#   genelleme yapamaması durumu. Tespit: CV AUC ile Test AUC
#   arasındaki fark (gap) > 0.05 ise şüphe uyanır.
#
# Kullanılan önlemler:
#   XGBoost   → min_child_weight, gamma (yaprak budama),
#               reg_alpha (L1), reg_lambda (L2), early_stopping
#   LightGBM  → min_data_in_leaf, lambda_l1, lambda_l2, early_stopping
#   RF         → max_depth, max_features='sqrt', min_samples_leaf
#
# Early Stopping:
#   Validation seti üzerinde performans iyileşmezse eğitimi durdurur.
#   CV (cross-validation) ile birlikte kullanılınca çakışma çıkar
#   (CV kendi eval_set'ini bilmez). Bu yüzden iki ayrı model nesnesi
#   oluşturuyoruz: biri early stopping'li, biri CV için.

# ─────────────────────────────────────────────────────────────
# NOT: Cross-validation için early_stopping_rounds OLMAYAN
# hafif modeller kullanılır (CV kendi eval_set'ini bilmez).
# Ana modeller ise validation seti ile early stopping yapar.
# ─────────────────────────────────────────────────────────────

# ══ Model 1: XGBoost ═════════════════════════════════════════
print("🚀 XGBoost eğitiliyor (regularized + early stopping)...")

# Paylaşılan hiperparametreler (ana model ve CV modeli aynı yapıyı kullanır)
_xgb_params = dict(
    max_depth=4,          # Ağaç derinliği sınırı → derin ağaç = daha fazla ezberleme
    learning_rate=0.03,   # Küçük adım → daha stabil öğrenme
    subsample=0.75,       # Her ağaç için rastgele %75 satır → varyasyonu azaltır
    colsample_bytree=0.75,# Her ağaç için rastgele %75 özellik
    min_child_weight=5,   # Bir yaprak oluşması için gereken min örnek ağırlığı
    gamma=0.2,            # Yaprak bölünmesi için gereken min kazanç (budama)
    reg_alpha=0.1,        # L1 düzenleme → seyrek (sparse) ağırlıklar
    reg_lambda=2.0,       # L2 düzenleme → büyük ağırlıkları cezalandırır
    random_state=42,
    eval_metric='auc',
    use_label_encoder=False
)

# Ana model: validation seti ile early stopping
xgb_model = xgb.XGBClassifier(n_estimators=500, early_stopping_rounds=30, **_xgb_params)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
best_xgb_rounds = xgb_model.best_iteration + 1

# CV modeli: early stopping YOK; best_iteration kadar ağaç kullanır
xgb_cv_model = xgb.XGBClassifier(n_estimators=best_xgb_rounds, **_xgb_params)
xgb_cv = cross_val_score(xgb_cv_model, X_train, y_train, cv=cv, scoring='roc_auc').mean()

xgb_pred  = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_acc   = accuracy_score(y_test, xgb_pred)
xgb_auc   = roc_auc_score(y_test, xgb_proba)
print(f"  ✓ Best rounds: {best_xgb_rounds} | Test AUC: {xgb_auc:.4f} | CV AUC: {xgb_cv:.4f} | Acc: {xgb_acc:.4f}")
print(f"  {'✅ İyi genelleme' if abs(xgb_auc - xgb_cv) < 0.05 else '⚠️ Overfitting riski'}: Test-CV gap = {abs(xgb_auc-xgb_cv):.4f}")

# ══ Model 2: Random Forest (Regularized) ═════════════════════
print("\n🌲 Random Forest eğitiliyor (regularized)...")
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,           # Ağaç derinliği sınırı
    max_features='sqrt',   # Her bölünmede sqrt(n_feature) özellik dener
    min_samples_leaf=10,   # Yaprakta en az 10 örnek zorunlu
    min_samples_split=20,  # Bölünme için en az 20 örnek zorunlu
    random_state=42,
    n_jobs=-1              # Tüm CPU çekirdeğini kullan
)
rf_model.fit(X_train, y_train)
rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_cv    = cross_val_score(rf_model, X_train, y_train, cv=cv, scoring='roc_auc').mean()
rf_acc   = accuracy_score(y_test, rf_pred)
rf_auc   = roc_auc_score(y_test, rf_proba)
print(f"  ✓ Test AUC: {rf_auc:.4f} | CV AUC: {rf_cv:.4f} | Acc: {rf_acc:.4f}")
print(f"  {'✅ İyi genelleme' if abs(rf_auc - rf_cv) < 0.05 else '⚠️ Overfitting riski'}: Test-CV gap = {abs(rf_auc-rf_cv):.4f}")

# ══ Model 3: LightGBM (Regularized + Early Stopping) ═════════
print("\n💡 LightGBM eğitiliyor (regularized + early stopping)...")

_lgb_params = dict(
    max_depth=4,
    learning_rate=0.03,
    num_leaves=15,         # 2^max_depth'ten küçük tutularak overfitting önlenir
    min_data_in_leaf=20,   # Yaprak başına min örnek sayısı
    feature_fraction=0.75, # Her iterasyonda rastgele özellik alt kümesi
    bagging_fraction=0.75, # Her iterasyonda rastgele satır alt kümesi
    bagging_freq=5,        # Her 5 iterasyonda bir bagging uygula
    lambda_l1=0.1,         # L1 düzenleme
    lambda_l2=2.0,         # L2 düzenleme
    random_state=42,
    verbose=-1             # Log mesajlarını bastır
)

# Ana model: early stopping ile
lgb_model = lgb.LGBMClassifier(n_estimators=500, **_lgb_params)
lgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
best_lgb_rounds = lgb_model.best_iteration_

# CV modeli: early stopping YOK
lgb_cv_model = lgb.LGBMClassifier(n_estimators=best_lgb_rounds, **_lgb_params)
lgb_cv = cross_val_score(lgb_cv_model, X_train, y_train, cv=cv, scoring='roc_auc').mean()

lgb_pred  = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_acc   = accuracy_score(y_test, lgb_pred)
lgb_auc   = roc_auc_score(y_test, lgb_proba)
print(f"  ✓ Best rounds: {best_lgb_rounds} | Test AUC: {lgb_auc:.4f} | CV AUC: {lgb_cv:.4f} | Acc: {lgb_acc:.4f}")
print(f"  {'✅ İyi genelleme' if abs(lgb_auc - lgb_cv) < 0.05 else '⚠️ Overfitting riski'}: Test-CV gap = {abs(lgb_auc-lgb_cv):.4f}")

# ── Model karşılaştırma tablosu ──────────────────────────────
model_comparison = pd.DataFrame({
    'Model':          ['XGBoost', 'Random Forest', 'LightGBM'],
    'Test AUC':       [xgb_auc, rf_auc, lgb_auc],
    'CV AUC (5-fold)':[xgb_cv, rf_cv, lgb_cv],
    'Gap (az=iyi)':   [round(abs(xgb_auc-xgb_cv),4), round(abs(rf_auc-rf_cv),4), round(abs(lgb_auc-lgb_cv),4)],
    'Accuracy':       [xgb_acc, rf_acc, lgb_acc]
}).sort_values('Test AUC', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("🏆 MODEL KARŞILAŞTIRMASI (CV Gap < 0.05 = overfitting yok)")
print("="*80)
display(model_comparison)

# En iyi modeli seç
best_model_name = model_comparison.iloc[0]['Model']
if best_model_name == 'XGBoost':
    best_model = xgb_model; best_pred = xgb_pred
elif best_model_name == 'Random Forest':
    best_model = rf_model;  best_pred = rf_pred
else:
    best_model = lgb_model; best_pred = lgb_pred

print(f"\n🥇 En İyi Model: {best_model_name}")
print(classification_report(y_test, best_pred, target_names=['Reject', 'Hire']))


🚀 XGBoost eğitiliyor (regularized + early stopping)...
  ✓ Best rounds: 11 | Test AUC: 0.9898 | CV AUC: 0.9794 | Acc: 0.8100
  ✅ İyi genelleme: Test-CV gap = 0.0104

🌲 Random Forest eğitiliyor (regularized)...
  ✓ Test AUC: 0.9946 | CV AUC: 0.9884 | Acc: 0.9750
  ✅ İyi genelleme: Test-CV gap = 0.0062

💡 LightGBM eğitiliyor (regularized + early stopping)...
  ✓ Best rounds: 264 | Test AUC: 0.9966 | CV AUC: 0.9979 | Acc: 0.9750
  ✅ İyi genelleme: Test-CV gap = 0.0014

🏆 MODEL KARŞILAŞTIRMASI (CV Gap < 0.05 = overfitting yok)


,Model,Test AUC,CV AUC (5-fold),Gap (az=iyi),Accuracy
0,LightGBM,0.996589,0.997949,0.0014,0.975
1,Random Forest,0.994639,0.988410,0.0062,0.975
2,XGBoost,0.989847,0.979436,0.0104,0.810



🥇 En İyi Model: LightGBM
              precision    recall  f1-score   support

      Reject       0.92      0.95      0.94        38
        Hire       0.99      0.98      0.98       162

    accuracy                           0.97       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.98      0.97      0.98       200



## 🔍 6. SHAP Açıklanabilirlik — Küresel Görünüm

In [40]:
# ============================================================
# SHAP AÇIKLANAB İLİRLİĞİ — KÜRESEL GÖRÜNÜM
# ============================================================
#
# SHAP (SHapley Additive exPlanations) nasıl çalışır?
#   Koalisyonel oyun teorisinden esinlenmiştir. Her özelliğin
#   modelin çıktısına katkısı, o özellik dahil/hariç bırakıldığında
#   ortaya çıkan fark ortalaması alınarak hesaplanır.
#
# TreeExplainer:
#   Ağaç tabanlı modeller (XGBoost, LightGBM, RF) için özel
#   optimize edilmiş SHAP algoritması. Kaba kuvvete kıyasla
#   çok daha hızlı çalışır (polinomiyal yerine lineer karmaşıklık).
#
# shap_values şekli:
#   [n_örnek, n_özellik] — her hücre o özelliğin o örnek için
#   model çıktısına katkısını gösterir.
#   Pozitif değer → işe alınma olasılığını artırır.
#   Negatif değer → olasılığı düşürür.

print("🔮 SHAP explainer hazırlanıyor...")

# TreeExplainer oluştur; özellik isimlerini Türkçe etiketlerle ver
explainer = shap.TreeExplainer(best_model, feature_names=feature_names_list)

# shap.Explanation nesnesi → waterfall/force plot için kullanılır
shap_values_test = explainer(X_test.rename(columns=feature_names_tr))

# Eski API uyumu: shap_values() → numpy dizisi döner
# İkili sınıflandırmada bazı modeller [sınıf0_shap, sınıf1_shap] döner;
# biz pozitif sınıfı (Hire=1) istiyoruz → index [1]
raw_shap = explainer.shap_values(X_test)
if isinstance(raw_shap, list):
    raw_shap = raw_shap[1]  # Pozitif sınıf SHAP değerleri

# ── Küresel Özet Grafiği (dot/beeswarm) ──────────────────────
# Her nokta bir adayı temsil eder.
# X ekseni: SHAP değeri (pozitif=hire yönünde etki)
# Renk: özellik değeri (kırmızı=yüksek, mavi=düşük)
fig_global, ax = plt.subplots(figsize=(11, 7))
shap.summary_plot(
    raw_shap,
    X_test.rename(columns=feature_names_tr),
    plot_type="dot",
    show=False,
    plot_size=None,
    axis_color='#333'
)
plt.title('🔍 SHAP Özellik Önem Dağılımı (Tüm Test Adayları)',
          fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_global.png', dpi=130, bbox_inches='tight')
plt.show()
print("✅ Küresel SHAP tamamlandı!")


🔮 SHAP explainer hazırlanıyor...
✅ Küresel SHAP tamamlandı!


In [41]:
# ─── Global Bar Önem Sırası ───
mean_shap = np.abs(raw_shap).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'Özellik': feature_names_list,
    'Ortalama |SHAP|': mean_shap
}).sort_values('Ortalama |SHAP|', ascending=True)

fig_bar, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(shap_importance_df)))
bars = ax.barh(shap_importance_df['Özellik'], shap_importance_df['Ortalama |SHAP|'], color=colors, edgecolor='none')
ax.set_xlabel('Ortalama |SHAP değeri| (model çıktısına etkisi)', fontsize=11)
ax.set_title('📊 Global Özellik Önemi (SHAP)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, shap_importance_df['Ortalama |SHAP|']):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_bar.png', dpi=130, bbox_inches='tight')
plt.show()

## 🎯 7. Geliştirilmiş İş İlanı Eşleştirme Sistemi

In [42]:
# ============================================================
# NLP TABANLI İŞ İLANI EŞLEŞTİRME SİSTEMİ
# ============================================================
#
# Bu hücrede iki ana bileşen var:
#
# 1) generate_shap_explanation_text()
#    SHAP sayısal değerlerini standart, ayrıntılı Türkçe rapora dönüştürür.
#    Her aday için aynı yapıda bir metin üretilir:
#      - Her özellik her zaman gösterilir (0 proje → "0 proje" yazar)
#      - Maaş beklentisi de rapora eklenir (opsiyonel parametre)
#      - Etki büyüklüğüne göre 4 seviye: 🔴🟠🟢💚
#      - Pozitif/negatif ayrımı açık şekilde belirtilir
#
# 2) ImprovedJobMatcher sınıfı
#    İş ilanı metni ile aday becerilerini karşılaştırır:
#      a) TF-IDF vektörizasyonu + kosinüs benzerliği  → Metinsel örtüşme skoru
#      b) Doğrudan anahtar kelime eşleştirmesi        → Beceri örtüşmesi
#      c) ML modelinin tahmin ettiği işe alınma olasılığı
#    Nihai Final Skor = %30 JD Eşleşme + %40 Model Olasılığı + %20 Deneyim + %10 Beceri Kapsamı
#
# TF-IDF Nasıl Çalışır?
#   Her belge sözcük frekansı vektörüne dönüştürülür.
#   Nadir ama anlamlı terimler (ör. "PyTorch") yüksek ağırlık alır.
#   cos(θ) = (A·B) / (|A|·|B|)  →  0=tamamen farklı, 1=aynı içerik
# ============================================================


def generate_shap_explanation_text(shap_vals, feature_vals,
                                   jd_matched=None, jd_missing=None,
                                   jd_coverage=None, jd_text=None,
                                   salary=None, candidate_skills=None):
    """
    Her aday için aynı yapıda, paragraf biçiminde Türkçe açıklama üretir.

    Çıktı yapısı (4 paragraf):
      1) Deneyim & proje portföyü
      2) Beceri seti (teknoloji bazlı, ilanla ilişkilendirilmiş)
      3) Eğitim, sertifikasyon ve maaş beklentisi
      4) Genel model kararı + ilan eşleşme özeti

    Her değer açıkça yazılır — 0 proje "0 proje", sertifikasyon yoksa "sertifikası bulunmuyor" gibi.
    SHAP etkisi parantez içinde (+/-) sayısal olarak gösterilir; renk ikonu eşliğinde.
    """

    # ── Teknoloji → ilan anahtar kelime eşlemesi ─────────────────────────────
    # İlanda bu anahtar kelimeler geçmiyorsa teknoloji "ilana dahil değil" sayılır
    tech_keywords = {
        "ML/AI Biliyor":         ["machine learning", "deep learning", "tensorflow", "pytorch",
                                   "ml", "ai ", "artificial intelligence", "neural"],
        "Python Biliyor":        ["python"],
        "SQL Biliyor":           ["sql", "database", "postgresql", "mysql"],
        "Cloud (AWS/GCP/Azure)": ["aws", "azure", "gcp", "cloud"],
        "Java Biliyor":          ["java", "kotlin", "jvm", "spring"],
        "Siber Güvenlik":        ["security", "cybersecurity", "penetration",
                                   "ethical hacking", "firewall", "oscp", "ceh"],
    }
    tech_display = {
        "ML/AI Biliyor":         "ML/AI",
        "Python Biliyor":        "Python",
        "SQL Biliyor":           "SQL",
        "Cloud (AWS/GCP/Azure)": "Cloud (AWS/Azure/GCP)",
        "Java Biliyor":          "Java",
        "Siber Güvenlik":        "Siber Güvenlik",
    }

    jd_lower = jd_text.lower() if jd_text else ""

    def tech_in_jd(feat):
        """Bu teknoloji iş ilanında geçiyor mu?"""
        if feat not in tech_keywords or not jd_lower:
            return True
        return any(kw in jd_lower for kw in tech_keywords[feat])

    # ── Özellik değerlerine sözlük erişimi ───────────────────────────────────
    feat_dict = dict(zip(feature_names_list, zip(shap_vals, feature_vals)))

    def get(feat):
        """(shap_val, feature_val) döndürür."""
        return feat_dict.get(feat, (0, 0))

    # ── SHAP etkisini okunabilir etikete çevir ────────────────────────────────
    # SHAP değeri ne kadar büyükse o özellik modelin kararını o kadar etkiliyor
    def etki(sv):
        """SHAP büyüklüğüne göre ikon + kısa etiket döndürür."""
        if sv >= 0.12:  return "💚", f"+{sv:.3f}"
        if sv >= 0.03:  return "🟢", f"+{sv:.3f}"
        if sv <= -0.12: return "🔴", f"{sv:.3f}"
        if sv <= -0.03: return "🟠", f"{sv:.3f}"
        return "⚪", f"{sv:.3f}"

    edu_map = {1: "lisans", 2: "yüksek lisans", 3: "doktora"}

    # ════════════════════════════════════════════════════════════════════════
    # PARAGRAF 1 — Deneyim & Proje
    # Her değer mutlaka yazılır; 0 da dahil.
    # ════════════════════════════════════════════════════════════════════════
    sv_exp,   fv_exp   = get("Deneyim (Yıl)")
    sv_proj,  fv_proj  = get("Proje Sayısı")
    sv_ratio, fv_ratio = get("Proje/Deneyim Oranı")

    # Deneyim seviyesini kelimeye çevir
    if fv_exp == 0:
        exp_sev = "hiç deneyimi yok (0 yıl)"
    elif fv_exp < 2:
        exp_sev = f"{fv_exp:.0f} yıl deneyimi var, başlangıç seviyesinde"
    elif fv_exp < 5:
        exp_sev = f"{fv_exp:.0f} yıl deneyimiyle orta düzey"
    elif fv_exp < 8:
        exp_sev = f"{fv_exp:.0f} yıl deneyimiyle kıdemli"
    else:
        exp_sev = f"{fv_exp:.0f} yıllık güçlü deneyime sahip"

    # Proje portföyünü kelimeye çevir
    if fv_proj == 0:
        proj_sev = "tamamlanmış projesi bulunmuyor"
    elif fv_proj < 3:
        proj_sev = f"{fv_proj:.0f} tamamlanmış projesiyle oldukça sınırlı bir portföye sahip"
    elif fv_proj < 6:
        proj_sev = f"{fv_proj:.0f} projelik temel bir portföy oluşturmuş"
    elif fv_proj < 10:
        proj_sev = f"{fv_proj:.0f} projelik iyi bir portföy sunuyor"
    else:
        proj_sev = f"{fv_proj:.0f} projelik güçlü bir portföy sergilemiş"

    # Verimlilik oranı
    if fv_ratio == 0:
        ratio_sev = "proje üretim verimliliği sıfır"
    elif fv_ratio < 0.5:
        ratio_sev = f"proje/deneyim oranı {fv_ratio:.2f} ile düşük verimlilik gösteriyor"
    elif fv_ratio < 1.5:
        ratio_sev = f"proje/deneyim oranı {fv_ratio:.2f} ile ortalama bir verimlilik sunuyor"
    else:
        ratio_sev = f"proje/deneyim oranı {fv_ratio:.2f} ile yüksek verimlilik gösteriyor"

    ik_e, sh_e = etki(sv_exp)
    ik_p, sh_p = etki(sv_proj)
    ik_r, sh_r = etki(sv_ratio)

    p1 = (
        f"Bu aday **{exp_sev}** {ik_e}({sh_e}). "
        f"{proj_sev.capitalize()} {ik_p}({sh_p}); {ratio_sev} {ik_r}({sh_r})."
    )

    # ════════════════════════════════════════════════════════════════════════
    # PARAGRAF 2 — Beceri Seti
    # Tüm teknolojiler gösterilir; ilanda gerekip gerekmediği belirtilir.
    # ════════════════════════════════════════════════════════════════════════
    sv_skill, fv_skill = get("Beceri Sayısı")
    ik_sk, sh_sk = etki(sv_skill)

    if fv_skill == 0:
        skill_sev = "hiç beceri tanımlanmamış"
    elif fv_skill < 4:
        skill_sev = f"yalnızca {fv_skill:.0f} beceriyle dar bir kapsama sahip"
    elif fv_skill < 7:
        skill_sev = f"{fv_skill:.0f} beceriyle orta düzey bir kapsama sahip"
    else:
        skill_sev = f"{fv_skill:.0f} beceriyle geniş bir yetenek yelpazesi sunuyor"

    # Her teknolojiyi grupla: ilanda gerekli + var, ilanda gerekli + yok, ilana dahil değil
    required_has   = []  # ilan gerektiriyor, adayda var
    required_lacks = []  # ilan gerektiriyor, adayda yok
    irrelevant     = []  # ilan gerektirmiyor

    tech_feats = ["Python Biliyor", "ML/AI Biliyor", "SQL Biliyor",
                  "Cloud (AWS/GCP/Azure)", "Java Biliyor", "Siber Güvenlik"]

    for feat in tech_feats:
        sv_t, fv_t = get(feat)
        ik_t, sh_t = etki(sv_t)
        dname = tech_display[feat]
        entry = (dname, fv_t, ik_t, sh_t)
        if tech_in_jd(feat):
            if fv_t == 1:
                required_has.append(entry)
            else:
                required_lacks.append(entry)
        else:
            irrelevant.append(entry)

    # Beceri paragrafını inşa et
    tech_parts = []
    if required_has:
        names = ", ".join(f"**{d}** {ik}({sh})" for d, _, ik, sh in required_has)
        tech_parts.append(f"ilanın aradığı {names} teknolojilerine hâkim")
    if required_lacks:
        names = ", ".join(f"**{d}** {ik}({sh})" for d, _, ik, sh in required_lacks)
        tech_parts.append(f"ancak ilanın gerektirdiği {names} bilgisi eksik")
    if irrelevant:
        names = ", ".join(f"**{d}**" for d, _, _, _ in irrelevant)
        has_irr  = [d for d, fv, _, _ in irrelevant if fv == 1]
        if has_irr:
            tech_parts.append(f"ilanın dışında da {', '.join('**' + d + '**' for d in has_irr)} bilgisi mevcut")

    tech_sentence = ("; ".join(tech_parts) + ".") if tech_parts else "teknoloji profili değerlendirilemedi."

    p2 = (
        f"Beceri profiline bakıldığında aday **{skill_sev}** {ik_sk}({sh_sk}). "
        f"Aday {tech_sentence}"
    )

    # ════════════════════════════════════════════════════════════════════════
    # PARAGRAF 3 — Eğitim, Sertifikasyon ve Maaş
    # ════════════════════════════════════════════════════════════════════════
    sv_edu,       fv_edu       = get("Eğitim Seviyesi")
    sv_cert_flag, fv_cert_flag = get("Sertifikası Var")
    sv_cert_cnt,  fv_cert_cnt  = get("Sertifika Sayısı")

    ik_edu, sh_edu = etki(sv_edu)
    ik_c,   sh_c   = etki(sv_cert_flag)

    edu_str = edu_map.get(int(fv_edu), "lisans")

    if fv_cert_cnt == 0:
        cert_sev = "sertifikası bulunmuyor"
    elif fv_cert_cnt == 1:
        cert_sev = "1 sertifikaya sahip"
    elif fv_cert_cnt < 4:
        cert_sev = f"{fv_cert_cnt:.0f} sertifikayla iyi bir sertifikasyon profiline sahip"
    else:
        cert_sev = f"{fv_cert_cnt:.0f} sertifikayla güçlü bir sertifikasyon geçmişine sahip"

    # Maaş beklentisi (model dışı bilgi, bilgilendirme amaçlı eklenir)
    if salary and salary > 0:
        if salary < 50000:
            maas_yorum = "piyasa altı bir beklenti"
        elif salary < 80000:
            maas_yorum = "piyasa ortası bir beklenti"
        elif salary < 120000:
            maas_yorum = "piyasa üstü bir beklenti"
        else:
            maas_yorum = "yüksek bir maaş beklentisi"
        maas_cumle = f" Maaş beklentisi **${salary:,.0f}** olup bu {maas_yorum} olarak değerlendirilebilir (puanlamaya dahil değildir)."
    else:
        maas_cumle = " Maaş beklentisi belirtilmemiştir."

    p3 = (
        f"Eğitim açısından **{edu_str}** mezunu olan aday {ik_edu}({sh_edu}), "
        f"{cert_sev} {ik_c}({sh_c}).{maas_cumle}"
    )

    # ════════════════════════════════════════════════════════════════════════
    # PARAGRAF 4 — Genel Değerlendirme & İlan Eşleşmesi
    # En güçlü pozitif ve negatif faktörlerin kısa özeti.
    # ════════════════════════════════════════════════════════════════════════
    all_feats_sorted = sorted(
        zip(feature_names_list, shap_vals, feature_vals),
        key=lambda x: x[1], reverse=True
    )
    top_pos = [(f, sv) for f, sv, _ in all_feats_sorted if sv >= 0.03][:2]
    top_neg = [(f, sv) for f, sv, _ in all_feats_sorted if sv <= -0.03][-2:]

    if top_pos:
        pos_str = " ve ".join(f"**{f}** ({sv:+.3f})" for f, sv in top_pos)
        guclendiren = f"modelin kararını en çok güçlendiren etkenler {pos_str}"
    else:
        guclendiren = "modelin kararını güçlendiren belirgin bir etken yok"

    if top_neg:
        neg_str = " ve ".join(f"**{f}** ({sv:+.3f})" for f, sv in top_neg)
        zayiflatan = f"kararı en çok zayıflatan etkenler ise {neg_str}"
    else:
        zayiflatan = "kararı olumsuz etkileyen belirgin bir faktör yok"

    # İlan eşleşme özeti
    if jd_coverage is not None:
        if jd_coverage >= 70:
            cov_yorum = "yüksek uyum"
        elif jd_coverage >= 40:
            cov_yorum = "orta uyum"
        else:
            cov_yorum = "düşük uyum"
        cov_cumle = f" İlan beceri kapsamı **%{jd_coverage:.0f}** ile {cov_yorum} gösteriyor."
    else:
        cov_cumle = ""

    if jd_matched:
        matched_cumle = f" Adayın `{', '.join(jd_matched[:4])}` becerileri ilanla örtüşüyor."
    else:
        matched_cumle = " İlanla örtüşen belirgin bir beceri tespit edilemedi."

    if jd_missing:
        missing_cumle = f" Öte yandan `{', '.join(jd_missing[:3])}` becerileri eksik görünüyor."
    else:
        missing_cumle = ""

    p4 = (
        f"Genel değerlendirmede {guclendiren}; {zayiflatan}."
        f"{cov_cumle}{matched_cumle}{missing_cumle}"
    )

    # ── Dört paragrafı ayraçla birleştir ─────────────────────────────────────
    return f"{p1}\n\n{p2}\n\n{p3}\n\n{p4}"

def get_candidate_shap_figure(candidate_features_df, hire_prob):
    """SHAP waterfall grafiği oluşturur."""
    X_cand = candidate_features_df[feature_cols].values
    cand_shap_raw = explainer.shap_values(candidate_features_df[feature_cols])
    if isinstance(cand_shap_raw, list):
        cand_shap_raw = cand_shap_raw[1]
    sv = cand_shap_raw[0]

    base_val = explainer.expected_value
    if isinstance(base_val, (list, np.ndarray)):
        base_val = base_val[1]

    exp_obj = shap.Explanation(
        values=sv,
        base_values=float(base_val),
        data=X_cand[0],
        feature_names=feature_names_list
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    shap.plots.waterfall(exp_obj, max_display=13, show=False)
    plt.title(
        f'SHAP Açıklaması — İşe Alınma Olasılığı: %{hire_prob:.1f}',
        fontsize=12, fontweight='bold', pad=12
    )
    plt.tight_layout()
    return fig


# ── İş ilanı metin tokenizasyonu için stop words ─────────────────────────────
_JD_STOPWORDS = {
    'and','the','for','with','our','you','are','that','this','have','will','must',
    'years','experience','knowledge','using','strong','good','ability','skills',
    'work','team','required','looking','join','need','plus','data','based',
    'level','senior','junior','mid','lead','engineer','developer','scientist',
    'manager','analyst','minimum','preferred','bonus','background','proven',
    'excellent','communication','written','verbal','related','field','degree',
    'bachelor','master','phd','equivalent','etc','including','such','other'
}


class ImprovedJobMatcher:
    """
    TF-IDF tabanlı iş ilanı–aday eşleştirme motoru.

    Nihai skor formülü (v4 — Deneyim ağırlıklı):
        Final = 0.30 × JD_Match + 0.40 × Hire_Prob + 0.20 × Exp_Score + 0.10 × Skill_Coverage
        (Exp_Score: 0–10 yıl arası normalize, 10+ yıl = 100 puan)
    """
    def __init__(self, df, model, scaler, feature_cols, role_skills_map):
        self.df            = df.copy()
        self.model         = model
        self.scaler        = scaler
        self.feature_cols  = feature_cols
        self.role_skills_map = role_skills_map

        # TF-IDF vectorizer: bigram (1-2 kelime), en fazla 200 terim
        self.vectorizer    = TfidfVectorizer(max_features=200, stop_words='english', ngram_range=(1, 2))
        self.skills_vectors = self.vectorizer.fit_transform(self.df['Skills'])

    def calculate_jd_match_score(self, job_description, candidate_skills):
        """
        İlan-aday eşleşme skorunu hesaplar (0–100).

        İki bileşenin ortalaması:
          1) TF-IDF kosinüs benzerliği   → semantik örtüşme
          2) Doğrudan beceri eşleşmesi   → kelime bazlı örtüşme
        """
        jd_vector        = self.vectorizer.transform([job_description])
        candidate_vector = self.vectorizer.transform([candidate_skills])
        tfidf_score      = cosine_similarity(jd_vector, candidate_vector)[0][0]

        jd_lower              = job_description.lower()
        candidate_skills_list = [s.strip().lower() for s in candidate_skills.split(',')]
        matched               = sum(1 for sk in candidate_skills_list if sk in jd_lower)
        direct_match          = matched / len(candidate_skills_list) if candidate_skills_list else 0

        return (tfidf_score * 0.5 + direct_match * 0.5) * 100

    def rank_candidates(self, job_description, job_role=None, top_n=20):
        """
        Tüm adayları sıralar; SHAP değerlerini ve JD eşleşme bilgilerini döndürür.
        """
        if job_role and job_role != 'Tümü':
            filtered_df = self.df[self.df['Job Role'] == job_role].copy()
        else:
            filtered_df = self.df.copy()

        # İlan metnini token kümesine çevir (stop words hariç)
        jd_tokens = set()
        for w in re.sub(r'[^a-z0-9#+.]', ' ', job_description.lower()).split():
            if len(w) > 2 and w not in _JD_STOPWORDS:
                jd_tokens.add(w)

        jd_scores, matched_lists, missing_lists, coverage_list = [], [], [], []
        for _, row in filtered_df.iterrows():
            score = self.calculate_jd_match_score(job_description, row['Skills'])
            jd_scores.append(score)

            cand_skills = [s.strip().lower() for s in str(row['Skills']).split(',')]
            matched = [s for s in cand_skills
                       if any(s in tok or tok in s for tok in jd_tokens if len(tok) > 2)]
            missing = [tok for tok in jd_tokens
                       if len(tok) > 2
                       and not any(tok in s or s in tok for s in cand_skills)][:4]
            coverage = len(matched) / max(len(jd_tokens), 1) * 100

            matched_lists.append(matched[:5])
            missing_lists.append(missing[:4])
            coverage_list.append(min(coverage, 100))

        filtered_df['JD_Match_Score'] = jd_scores
        filtered_df['_jd_matched']    = matched_lists
        filtered_df['_jd_missing']    = missing_lists
        filtered_df['_jd_coverage']   = coverage_list
        filtered_df['_jd_text']       = job_description

        X_features = filtered_df[self.feature_cols]
        filtered_df['Hire_Probability'] = self.model.predict_proba(X_features)[:, 1] * 100
        # ── Geliştirilmiş Final Skor Formülü ──────────────────────────────────
        # %30 JD Eşleşme (TF-IDF + direkt eşleşme)
        # %40 Model Olasılığı (işe alınma tahmini)
        # %20 Deneyim skoru (10 yıl üstü = 1.0; normalize edilmiş)
        # %10 Beceri kapsamı (ilanla örtüşen beceri oranı)
        exp_score = (filtered_df['Experience (Years)'].clip(upper=10) / 10) * 100
        skill_cov = filtered_df['_jd_coverage']  # zaten 0-100 arası
        filtered_df['Final_Score'] = (
            filtered_df['JD_Match_Score'] * 0.30 +
            filtered_df['Hire_Probability'] * 0.40 +
            exp_score * 0.20 +
            skill_cov * 0.10
        )

        shap_raw = explainer.shap_values(X_features)
        if isinstance(shap_raw, list):
            shap_raw = shap_raw[1]
        filtered_df['_shap_vals'] = list(shap_raw)
        filtered_df['_feat_vals'] = list(X_features.values)

        return filtered_df.sort_values('Final_Score', ascending=False).head(top_n)


matcher = ImprovedJobMatcher(df_processed, best_model, scaler, feature_cols, role_skills_map)
print("✅ Matcher v4 hazır!")


✅ Matcher v4 hazır!


## ➕ 8. Manuel Aday Ekleme

In [43]:
# ============================================================
# MANUEL ADAY EKLEME FONKSİYONU
# ============================================================
# Veri setinde yer almayan yeni adayları sisteme ekler.
# Adayın ham girdilerinden (beceriler, eğitim, deneyim...)
# modelin beklediği tüm mühendislik özelliklerini türetir.

def add_manual_candidate(name, skills, experience, education, certifications,
                         job_role, projects_count, salary_expectation):
    """
    Yeni bir aday satırı oluşturur ve df_processed formatında döndürür.

    Parametreler
    -----------
    name               : Aday adı soyadı
    skills             : Virgülle ayrılmış beceri dizesi (ör. 'Python, SQL, AWS')
    experience         : Deneyim yılı (int)
    education          : Eğitim kodu — 'B.Sc', 'B.Tech', 'MBA', 'M.Tech', 'PhD'
    certifications     : Virgülle ayrılmış sertifika dizesi (boş → 'None')
    job_role           : Başvurulan pozisyon adı
    projects_count     : Tamamlanan proje sayısı (int)
    salary_expectation : Beklenen maaş ($)

    Döndürür
    --------
    dict : df_processed ile aynı sütun yapısına sahip satır sözlüğü
    """

    # ── Temel bilgiler ──────────────────────────────────────────
    nc = {
        'Resume_ID':             df_processed['Resume_ID'].max() + 1,
        'Name':                  name,
        'Skills':                skills,
        'Experience (Years)':    experience,
        'Education':             education,
        'Certifications':        certifications if certifications else 'None',
        'Job Role':              job_role,       # ← Kullanıcının seçtiği pozisyon
        'Recruiter Decision':    'Pending',      # Henüz işe alım kararı yok
        'Salary Expectation ($)':salary_expectation,
        'Projects Count':        projects_count,
        'AI Score (0-100)':      0               # Kullanılmıyor; sütun uyumu için
    }

    # ── Özellik mühendisliği (Cell 9 ile aynı mantık) ──────────
    # Beceri sayısı
    nc['Skills_Count']  = len(skills.split(','))

    # Teknoloji varlık özellikleri (0/1)
    nc['Has_Python']    = int('python' in skills.lower())
    nc['Has_ML']        = int(any(t in skills.lower() for t in
                                  ['machine learning', 'deep learning', 'tensorflow', 'pytorch']))
    nc['Has_SQL']       = int('sql' in skills.lower())
    nc['Has_Cloud']     = int(any(t in skills.lower() for t in ['aws', 'azure', 'gcp', 'cloud']))
    nc['Has_Java']      = int('java' in skills.lower())
    nc['Has_Security']  = int(any(t in skills.lower() for t in
                                  ['security', 'ethical hacking', 'cybersecurity']))

    # Sertifikasyon özellikleri
    nc['Has_Certification'] = int(certifications and certifications.lower() != 'none')
    nc['Cert_Count']        = (0 if not certifications or certifications.lower() == 'none'
                               else len(certifications.split(',')))

    # Eğitim seviyesi ordinal skoru
    nc['Education_Score'] = education_scores.get(education, 1)

    # Proje verimliliği oranı
    nc['Experience_Project_Ratio'] = projects_count / (experience + 1)

    # Hedef: yeni adaylar için bilinmiyor; 0 olarak işaretlenir (model tahmini kullanılır)
    nc['Target'] = 0

    return nc

print("✅ Manuel aday fonksiyonu hazır!")


✅ Manuel aday fonksiyonu hazır!


## 🎨 9. Geliştirilmiş Gradio Arayüzü — Tabs + SHAP Paneli

In [44]:
# ============================================================
# GRADİO YARDIMCI FONKSİYONLARI
# ============================================================

# Sıralama sonuçlarını global sakla (SHAP sekmesi erişir)
_last_ranked = {}


def _find_candidate(ranked, name_input):
    """Adayı büyük/küçük harf gözetmeksizin, tam veya kısmi eşleşmeyle bulur."""
    name_q = name_input.strip().lower()
    exact = ranked[ranked['Name'].str.strip().str.lower() == name_q]
    if not exact.empty:
        return exact
    contains = ranked[ranked['Name'].str.strip().str.lower().str.contains(name_q, regex=False)]
    if not contains.empty:
        return contains
    return pd.DataFrame()


def evaluate_candidates_interface(job_description, job_role, top_n, manual_candidates_text):
    """
    Ana değerlendirme motoru.
    Manuel aday metnini ayrıştırır, sıralama yapar, sonuçları döndürür.
    """
    global _last_ranked
    temp_df = df_processed.copy()
    manual_names = []

    if manual_candidates_text and manual_candidates_text.strip():
        for line in manual_candidates_text.strip().split('\n'):
            if not line.strip() or line.startswith('#'):
                continue
            try:
                parts = [p.strip() for p in line.split('|')]
                if len(parts) < 7:
                    continue
                name, skills, exp, edu, cert, proj, salary = parts[:7]
                # Pozisyon: kullanıcının seçtiği rol; 'Tümü' ise ilk alfabetik role ata
                mc_role = job_role if job_role != 'Tümü' else sorted(role_skills_map.keys())[0]
                mc = add_manual_candidate(
                    name=name, skills=skills, experience=int(exp),
                    education=edu, certifications=cert,
                    job_role=mc_role,
                    projects_count=int(proj), salary_expectation=int(salary)
                )
                temp_df = pd.concat([temp_df, pd.DataFrame([mc])], ignore_index=True)
                manual_names.append(name)
            except Exception:
                continue

    temp_matcher = ImprovedJobMatcher(temp_df, best_model, scaler, feature_cols, role_skills_map)

    try:
        ranked = temp_matcher.rank_candidates(job_description, job_role, int(top_n))
    except Exception as e:
        return f"❌ Hata: {str(e)}", None, None

    _last_ranked = {
        'ranked':       ranked,
        'manual_names': manual_names,
        'jd_text':      job_description
    }

    # ── Özet metni ──
    summary = f"""### Değerlendirme Tamamlandı

**{len(ranked)} aday** sıralandı &nbsp;·&nbsp; 🆕 {len(manual_names)} yeni &nbsp;·&nbsp; {len(ranked)-len(manual_names)} mevcut

| Metrik | Değer |
|---|---|
| Ort. İlan Eşleşmesi | %{ranked['JD_Match_Score'].mean():.1f} |
| Ort. İşe Alınma Olasılığı | %{ranked['Hire_Probability'].mean():.1f} |
| En Yüksek Final Skoru | {ranked['Final_Score'].max():.1f} |

### İlk 5 Aday\n"""

    for i, (idx, cand) in enumerate(ranked.head(5).iterrows()):
        badge = "🆕" if cand['Name'] in manual_names else "📋"
        # Maaş bilgisini salary parametresiyle geç
        shap_text = generate_shap_explanation_text(
            cand['_shap_vals'], cand['_feat_vals'],
            jd_matched=cand.get('_jd_matched', []),
            jd_missing=cand.get('_jd_missing', []),
            jd_coverage=cand.get('_jd_coverage', None),
            jd_text=job_description,
            salary=cand.get('Salary Expectation ($)', None)
        )
        summary += f"""
**{i+1}. {badge} {cand['Name']}** &nbsp;·&nbsp; Final: **{cand['Final_Score']:.1f}** &nbsp;·&nbsp; Eşleşme: {cand['JD_Match_Score']:.1f}% &nbsp;·&nbsp; İşe Alınma: {cand['Hire_Probability']:.1f}%

{shap_text}

---"""

    # ── Sıralama tablosu ──
    result_df = ranked[[
        'Name', 'Job Role', 'Experience (Years)', 'Education',
        'Projects Count', 'JD_Match_Score', 'Hire_Probability', 'Final_Score'
    ]].copy()
    result_df.columns = [
        'İsim', 'Pozisyon', 'Deneyim (Yıl)', 'Eğitim', 'Proje Sayısı',
        'İlan Eşl. %', 'İşe Alınma %', 'Final Skor'
    ]
    result_df['Final Skor']   = result_df['Final Skor'].round(1)
    result_df['İlan Eşl. %']  = result_df['İlan Eşl. %'].round(1)
    result_df['İşe Alınma %'] = result_df['İşe Alınma %'].round(1)

    if manual_names:
        result_df.insert(0, 'Tür', result_df['İsim'].apply(
            lambda x: '🆕 YENİ' if x in manual_names else '📋 Mevcut'))

    # ── Plotly grafik ──
    top10 = ranked.head(10)
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=top10['JD_Match_Score'], y=top10['Name'], orientation='h',
        name='İlan Eşleşmesi %',
        marker_color='rgba(34,197,94,0.75)', width=0.35, offset=-0.2
    ))
    fig.add_trace(go.Bar(
        x=top10['Hire_Probability'], y=top10['Name'], orientation='h',
        name='İşe Alınma %',
        marker_color='rgba(59,130,246,0.75)', width=0.35, offset=0.2
    ))
    fig.add_trace(go.Scatter(
        x=top10['Final_Score'], y=top10['Name'],
        mode='markers+text', name='Final Skor',
        marker=dict(symbol='diamond', size=10, color='rgba(239,68,68,0.9)'),
        text=top10['Final_Score'].round(1),
        textposition='middle right', textfont=dict(size=9)
    ))
    fig.update_layout(
        title='Top 10 Aday — İlan Eşleşmesi / İşe Alınma / Final Skor',
        xaxis_title='Skor (%)', xaxis_range=[0, 115],
        barmode='overlay', height=500,
        yaxis={'categoryorder': 'total ascending'},
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        margin=dict(l=140, r=70, t=70, b=50),
        font=dict(family='DM Sans, sans-serif')
    )
    return summary, result_df, fig


def show_candidate_shap(candidate_name):
    """SHAP sekmesi: bireysel aday waterfall grafiği + açıklama tablosu."""
    global _last_ranked
    if not _last_ranked:
        return None, "⚠️ Önce Değerlendirme sekmesinden adayları sıralayın."

    ranked  = _last_ranked['ranked']
    jd_text = _last_ranked.get('jd_text', '')

    row = _find_candidate(ranked, candidate_name)
    if row.empty:
        names = ranked['Name'].tolist()
        liste = "\n".join(f"- {n}" for n in names[:15])
        return None, (
            f"❌ **'{candidate_name}'** sıralamada bulunamadı.\n\n"
            f"**Sıralamadaki adaylar:**\n{liste}"
        )

    cand        = row.iloc[0]
    hire_prob   = cand['Hire_Probability']
    shap_vals   = cand['_shap_vals']
    feat_vals   = cand['_feat_vals']
    jd_matched  = cand.get('_jd_matched', [])
    jd_missing  = cand.get('_jd_missing', [])
    jd_coverage = cand.get('_jd_coverage', None)

    fig = get_candidate_shap_figure(row[feature_cols], hire_prob)

    # Maaş ve beceri bilgisini de geç
    paragraph = generate_shap_explanation_text(
        shap_vals, feat_vals,
        jd_matched=jd_matched,
        jd_missing=jd_missing,
        jd_coverage=jd_coverage,
        jd_text=jd_text,
        salary=cand.get('Salary Expectation ($)', None)
    )

    detail  = f"### {cand['Name']} — SHAP Analizi\n\n"
    detail += f"**İşe Alınma Olasılığı:** %{hire_prob:.1f}"
    if jd_coverage is not None:
        detail += f" &nbsp;·&nbsp; **İlan Beceri Kapsamı:** %{jd_coverage:.0f}"
    detail += f"\n\n{paragraph}\n\n---\n\n"

    tech_keywords = {
        "ML/AI Biliyor":         ["machine learning","deep learning","tensorflow","pytorch","ml","ai ","neural"],
        "Python Biliyor":        ["python"],
        "SQL Biliyor":           ["sql","database","postgresql","mysql"],
        "Cloud (AWS/GCP/Azure)": ["aws","azure","gcp","cloud"],
        "Java Biliyor":          ["java","kotlin","jvm","spring"],
        "Siber Güvenlik":        ["security","cybersecurity","penetration","ethical hacking","firewall"],
    }
    jd_lower = jd_text.lower()

    def feat_relevant(feat):
        if feat not in tech_keywords:
            return True
        if not jd_lower:
            return True
        return any(kw in jd_lower for kw in tech_keywords[feat])

    contributions = sorted(
        zip(feature_names_list, shap_vals, feat_vals),
        key=lambda x: abs(x[1]), reverse=True
    )

    detail += "| Özellik | Değer | SHAP Etkisi | Yorum |\n|---|---|---|---|\n"
    for feat, sv, fv in contributions:
        if not feat_relevant(feat):
            continue
        yorum = "🟢 Artırıyor" if sv > 0.01 else ("🔴 Düşürüyor" if sv < -0.01 else "⚪ Etkisiz")
        detail += f"| {feat} | {fv:.2f} | {sv:+.4f} | {yorum} |\n"

    if jd_matched:
        detail += f"\n**✅ İlanda eşleşen beceriler:** `{', '.join(jd_matched)}`\n"
    if jd_missing:
        detail += f"**❌ İlanda eksik beceriler:** `{', '.join(jd_missing)}`\n"

    return fig, detail


# ── Aday havuzu yönetimi ────────────────────────────────────────────────────

def build_manual_text(name, skills, exp, edu, cert, proj, salary):
    """Form alanlarını pipe-format satırına dönüştürür."""
    if not name.strip():
        return ""
    cert_val = cert.strip() if cert.strip() else "None"
    return f"{name} | {skills} | {int(exp)} | {edu} | {cert_val} | {int(proj)} | {int(salary)}"


# BUG FIX: job_role parametresi eklendi.
# Önceki imza: add_candidate_to_pool(name, skills, exp, edu, cert, proj, salary, current_list)
# Yeni imza:   add_candidate_to_pool(name, skills, exp, edu, cert, proj, salary, current_list, job_role)
# Gradio add_btn.click inputs listesinde de job_role sona eklendi (Cell 22).
def add_candidate_to_pool(name, skills, exp, edu, cert, proj, salary, current_list, job_role):
    """
    Formdaki adayı birikimli metin havuzuna ekler.
    job_role: Filtreler bölümündeki pozisyon seçimi — adayın 'Job Role' sütununu doğru doldurur.
    """
    line = build_manual_text(name, skills, exp, edu, cert, proj, salary)
    if not line:
        return current_list, current_list, gr.update(value="⚠️ Lütfen aday adını girin.")

    existing = current_list.strip().split("\n") if current_list.strip() else []
    existing = [l for l in existing if not l.startswith("#")]
    existing.append(line)
    pool_text = "\n".join(existing)
    count  = len(existing)
    status = f"✅ **{name}** eklendi — Toplam **{count}** manuel aday hazır."
    return pool_text, pool_text, gr.update(value=status)


def clear_pool():
    """Aday havuzunu tamamen temizler."""
    return "", "", gr.update(value="🗑️ Liste temizlendi.")


print("✅ Gradio fonksiyonları hazır!")


✅ Gradio fonksiyonları hazır!


In [45]:
# ─── YENİ GRADIO ARAYÜZÜ (v4 - Temiz & Bölünmüş Düzen) ───

CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&display=swap');

*, *::before, *::after { box-sizing: border-box; }

body, .gradio-container { font-family: 'DM Sans', sans-serif !important; }

/* ─── Header ─── */
.app-header {
    background: #0F172A;
    color: white;
    padding: 20px 28px;
    border-radius: 14px;
    margin-bottom: 0;
    display: flex;
    align-items: center;
    justify-content: space-between;
}
.app-header .title { font-size: 1.25rem; font-weight: 600; letter-spacing: -0.01em; margin: 0; }
.app-header .sub   { font-size: 0.8rem; opacity: 0.5; margin: 3px 0 0; }
.app-header .badge {
    background: #334155; color: #94A3B8;
    font-size: 0.72rem; padding: 4px 10px;
    border-radius: 20px; font-weight: 500;
}

/* ─── Panel başlıkları ─── */
.panel-label {
    font-size: 0.72rem;
    font-weight: 600;
    letter-spacing: 0.07em;
    text-transform: uppercase;
    color: #64748B;
    margin: 0 0 10px 2px;
}

/* ─── Kartlar ─── */
.section-card {
    background: white;
    border: 1px solid #E2E8F0;
    border-radius: 12px;
    padding: 18px 20px;
    margin-bottom: 12px;
}

/* ─── Şablon butonları ─── */
.tpl-btn {
    background: #F8FAFC !important;
    border: 1px solid #E2E8F0 !important;
    border-radius: 8px !important;
    color: #475569 !important;
    font-size: 0.8rem !important;
    font-weight: 500 !important;
    padding: 7px 12px !important;
    transition: all 0.15s !important;
}
.tpl-btn:hover {
    background: #EFF6FF !important;
    border-color: #BFDBFE !important;
    color: #1D4ED8 !important;
}

/* ─── Ana buton ─── */
.run-btn {
    background: #1E293B !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-size: 0.95rem !important;
    font-weight: 600 !important;
    padding: 13px !important;
    width: 100% !important;
    letter-spacing: 0.01em !important;
    transition: background 0.15s !important;
}
.run-btn:hover { background: #0F172A !important; }

/* ─── SHAP butonu ─── */
.shap-btn {
    background: #0D9488 !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    font-size: 0.85rem !important;
}

/* ─── Sonuç kartı ─── */
.result-box {
    background: #F8FAFC;
    border: 1px solid #E2E8F0;
    border-radius: 12px;
    padding: 20px;
}

/* ─── Aday form alanları ─── */
.candidate-form label { font-size: 0.82rem !important; color: #475569 !important; font-weight: 500 !important; }
.candidate-form input, .candidate-form textarea, .candidate-form select {
    border-radius: 8px !important;
    border: 1px solid #E2E8F0 !important;
    font-size: 0.88rem !important;
}

/* ─── İpucu metni ─── */
.hint {
    font-size: 0.76rem;
    color: #94A3B8;
    margin: 5px 0 0 2px;
}

/* ─── Sekmeler ─── */
.tabs > .tab-nav button {
    font-size: 0.85rem !important;
    font-weight: 500 !important;
    border-radius: 8px 8px 0 0 !important;
}

/* ─── Divider ─── */
.divider { border: none; border-top: 1px solid #E2E8F0; margin: 12px 0; }

/* ─── Sol/sağ sütun ayırıcı ─── */
.left-col  { border-right: 1px solid #E2E8F0; padding-right: 16px !important; }
.right-col { padding-left: 16px !important; }

footer { display: none !important; }
"""

EXAMPLE_JD_AI = """AI Researcher / Machine Learning Engineer

Requirements:
- 3+ years of experience in AI/ML research or applied ML
- Deep knowledge of TensorFlow, PyTorch, NLP techniques
- Publications or strong project portfolio in ML/AI
- Python proficiency; experience with neural network architectures
- Familiarity with MLOps and experiment tracking tools
"""

EXAMPLE_JD_DS = """We are looking for a Senior Data Scientist.

Requirements:
- 5+ years of experience in data science or machine learning
- Proficiency in Python, SQL, TensorFlow or PyTorch
- Experience with cloud platforms (AWS, GCP or Azure)
- Strong background in statistical modeling
- Excellent communication skills"""

EXAMPLE_JD_MOBILE = """Senior Software Engineer

Requirements:
- 4+ years of software development experience
- Proficiency in Java, C++, React or similar languages
- Experience with SQL databases and REST APIs
- Agile/Scrum methodology
- Strong problem-solving and system design skills"""

EXAMPLE_JD_SECURITY = """Cybersecurity Engineer

- 3+ years in information security
- Ethical Hacking, Penetration Testing
- Network security & firewall configuration
- Security certifications (CEH, OSCP preferred)
- Python scripting for automation"""

# ─── Ana arayüz ───
with gr.Blocks(
    title="Aday Değerlendirme Sistemi",
    theme=gr.themes.Soft(
        primary_hue="slate",
        secondary_hue="blue",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("DM Sans")
    ),
    css=CSS
) as demo:

    # ════ HEADER ════
    gr.HTML("""
    <div class="app-header">
        <div>
            <p class="title">Aday Değerlendirme Sistemi</p>
            <p class="sub">XAI destekli işe alım · SHAP açıklamaları</p>
        </div>
        <span class="badge">v3 · XAI Enhanced</span>
    </div>
    """)

    with gr.Tabs():

        # ══════════════════════════════════════════════════
        # TAB 1: ANA DEĞERLENDİRME
        # ══════════════════════════════════════════════════
        with gr.Tab("Değerlendirme"):

            with gr.Row(equal_height=False):

                # ╔══════════════════════════════╗
                # ║   SOL PANEL: GİRİŞLER        ║
                # ╚══════════════════════════════╝
                with gr.Column(scale=4, elem_classes="left-col"):

                    # ── Bölüm 1: İş İlanı ──
                    gr.HTML('<p class="panel-label">İş İlanı</p>')
                    with gr.Group(elem_classes="section-card"):
                        job_description = gr.Textbox(
                            label="Pozisyon tanımı",
                            placeholder="Pozisyon gereksinimlerini buraya yapıştırın...",
                            lines=7,
                            show_copy_button=True
                        )
                        with gr.Row():
                            btn_ai  = gr.Button("AI Researcher",     size="sm", elem_classes="tpl-btn")
                            btn_ds  = gr.Button("Data Scientist",    size="sm", elem_classes="tpl-btn")
                            btn_mob = gr.Button("SW Engineer",       size="sm", elem_classes="tpl-btn")
                            btn_sec = gr.Button("Siber Güvenlik",    size="sm", elem_classes="tpl-btn")
                        gr.HTML('<p class="hint">Şablon seçince ilan alanı otomatik dolar</p>')

                        btn_ai.click( fn=lambda: EXAMPLE_JD_AI,       outputs=[job_description])
                        btn_ds.click( fn=lambda: EXAMPLE_JD_DS,       outputs=[job_description])
                        btn_mob.click(fn=lambda: EXAMPLE_JD_MOBILE,   outputs=[job_description])
                        btn_sec.click(fn=lambda: EXAMPLE_JD_SECURITY, outputs=[job_description])

                    # ── Bölüm 2: Filtreler ──
                    gr.HTML('<p class="panel-label" style="margin-top:4px">Filtreler</p>')
                    with gr.Group(elem_classes="section-card"):
                        with gr.Row():
                            job_role = gr.Dropdown(
                                label="Pozisyon",
                                choices=['Tümü'] + sorted(role_skills_map.keys()),
                                value='Tümü'
                            )
                            top_n = gr.Slider(
                                label="Gösterilecek aday sayısı",
                                minimum=5, maximum=50, value=10, step=5
                            )
                        required_skills_md = gr.Markdown(
                            "Pozisyon seçince önerilen beceriler görünür.",
                            elem_classes="hint"
                        )
                        def show_req_skills(role):
                            if role == 'Tümü':
                                return "Pozisyon seçince önerilen beceriler görünür."
                            skills = role_skills_map.get(role, [])
                            return f"**{role} için önerilen beceriler:** {', '.join(skills[:8])}"
                        job_role.change(show_req_skills, inputs=[job_role], outputs=[required_skills_md])

                    # ── Bölüm 3: Manuel Aday Girişi ──
                    gr.HTML('<p class="panel-label" style="margin-top:4px">Manuel Aday Ekle</p>')
                    with gr.Group(elem_classes="section-card candidate-form"):
                        with gr.Row():
                            c_name   = gr.Textbox(label="Ad Soyad",   placeholder="Ayşe Demir", scale=2)
                            c_edu    = gr.Dropdown(
                                label="Eğitim",
                                choices=["B.Sc","B.Tech","MBA","M.Tech","PhD"],
                                value="B.Tech", scale=1
                            )
                        c_skills = gr.Textbox(
                            label="Beceriler (virgülle ayırın)",
                            placeholder="Python, TensorFlow, SQL, AWS",
                            lines=2
                        )
                        with gr.Row():
                            c_exp    = gr.Number(label="Deneyim (yıl)", value=3, minimum=0, maximum=30, step=1, scale=1)
                            c_proj   = gr.Number(label="Proje sayısı",  value=5, minimum=0, maximum=50, step=1, scale=1)
                            c_salary = gr.Number(label="Maaş beklentisi ($)", value=70000, step=5000, scale=2)
                        c_cert = gr.Textbox(
                            label="Sertifikalar (opsiyonel, virgülle ayırın)",
                            placeholder="AWS Certified, Google ML ..."
                        )

                        with gr.Row():
                            add_btn   = gr.Button("+ Adayı Listeye Ekle", variant="secondary", size="sm")
                            clear_btn = gr.Button("Listeyi Temizle",        variant="stop",      size="sm")

                        add_status = gr.Markdown("", elem_classes="hint")

                        # Gizli: biriken aday listesi (pipe format)
                        manual_pool_state = gr.State("")
                        manual_pool_display = gr.Textbox(
                            label="Eklenmiş adaylar (düzenlenebilir)",
                            lines=4,
                            value="",
                            placeholder="Buraya henüz aday eklenmedi..."
                        )

                        # BUG FIX: inputs listesine job_role eklendi
                        # Fonksiyon imzasıyla eşleşmesi için 9. argüman olarak gönderilir.
                        add_btn.click(
                            fn=add_candidate_to_pool,
                            inputs=[c_name, c_skills, c_exp, c_edu, c_cert, c_proj, c_salary, manual_pool_state, job_role],
                            outputs=[manual_pool_state, manual_pool_display, add_status]
                        )
                        clear_btn.click(
                            fn=clear_pool,
                            outputs=[manual_pool_state, manual_pool_display, add_status]
                        )

                    # ── Değerlendir Butonu ──
                    evaluate_btn = gr.Button(
                        "Adayları Değerlendir →",
                        variant="primary", size="lg",
                        elem_classes="run-btn"
                    )

                # ╔══════════════════════════════╗
                # ║  SAĞ PANEL: SONUÇLAR         ║
                # ╚══════════════════════════════╝
                with gr.Column(scale=6, elem_classes="right-col"):

                    gr.HTML('<p class="panel-label">Değerlendirme Sonuçları</p>')

                    with gr.Group(elem_classes="result-box"):
                        summary_out = gr.Markdown(
                            "**Değerlendirme bekleniyor...**\nSol panelden iş ilanı girin ve butona basın.",
                        )

                    chart_out = gr.Plot(label="Aday Skor Karşılaştırması")

                    with gr.Group(elem_classes="result-box"):
                        gr.Markdown("#### Tüm Aday Sıralaması")
                        table_out = gr.Dataframe(
                            wrap=True,
                            interactive=False,
                            row_count=(10, "dynamic")
                        )

                    gr.HTML("""
                    <p class="hint" style="text-align:center; margin-top:8px">
                    Final Skor = %50 İlan Eşleşme (TF-IDF + beceri örtüşmesi) + %50 ML Modeli
                    &nbsp;·&nbsp; AI Score kullanılmıyor
                    </p>
                    """)

            # ── Bağlantı ──
            evaluate_btn.click(
                fn=evaluate_candidates_interface,
                inputs=[job_description, job_role, top_n, manual_pool_state],
                outputs=[summary_out, table_out, chart_out]
            )

        # ══════════════════════════════════════════════════
        # TAB 2: SHAP AÇIKLAMASI
        # ══════════════════════════════════════════════════
        with gr.Tab("SHAP Açıklaması"):

            gr.HTML("""
            <div style="background:#F0FDF4;border:1px solid #BBF7D0;border-radius:10px;
                        padding:14px 18px;margin-bottom:12px;font-size:0.86rem;color:#166534">
                Önce <strong>Değerlendirme</strong> sekmesinde adayları sıralayın,
                ardından burada aday adını yazıp açıklama alın.<br>
                <span style="opacity:0.7">
                    Yeşil SHAP → işe almayı destekler &nbsp;·&nbsp;
                    Kırmızı SHAP → işe almayı engeller &nbsp;·&nbsp;
                    Gri → etkisiz
                </span>
            </div>
            """)

            with gr.Row():
                candidate_name_input = gr.Textbox(
                    label="Aday adı",
                    placeholder="Örn: Ayşe Demir",
                    scale=5
                )
                shap_btn = gr.Button("SHAP ile Açıkla", variant="secondary",
                                     scale=1, elem_classes="shap-btn")

            with gr.Row():
                with gr.Column(scale=5):
                    shap_plot_out = gr.Plot(label="SHAP Waterfall Chart")
                with gr.Column(scale=5):
                    shap_detail_out = gr.Markdown(label="Detaylı Tablo")

            shap_btn.click(
                fn=show_candidate_shap,
                inputs=[candidate_name_input],
                outputs=[shap_plot_out, shap_detail_out]
            )

        # ══════════════════════════════════════════════════
        # TAB 3: KÜRESEL XAI
        # ══════════════════════════════════════════════════
        with gr.Tab("Küresel XAI"):

            with gr.Row():
                with gr.Column(scale=5):
                    gr.Markdown(f"""
**Model:** {best_model_name}
**Regularization:** Early Stopping + L1/L2 + min_child_weight
**Doğrulama:** 5-Fold Stratified Cross-Validation
**Özellik sayısı:** {len(feature_cols)} (AI Score dahil değil)

---

**SHAP nedir?**
Her özelliğin model kararına katkısını oyun teorisiyle hesaplar.
Pozitif değer işe almayı destekler, negatif engeller.
                    """)

                    gr.Markdown("#### Özellik Açıklamaları")
                    gr.Markdown("""
| Özellik | Açıklama |
|---|---|
| Deneyim (Yıl) | Toplam iş deneyimi |
| Proje Sayısı | Tamamlanan proje adedi |
| Beceri Sayısı | Listedeki toplam beceri |
| Python/ML/SQL/Cloud | Teknoloji bilgisi (0/1) |
| Java/Siber Güvenlik | Teknoloji bilgisi (0/1) |
| Sertifika Var / Sayısı | Sertifikasyon durumu |
| Eğitim Seviyesi | B.Sc=1 · MBA/M.Tech=2 · PhD=3 |
| Proje/Deneyim Oranı | Deneyim başına üretkenlik |
                    """)

                with gr.Column(scale=5):
                    gr.Image(
                        value='/kaggle/working/shap_global.png',
                        label="SHAP Küresel Dağılım",
                        show_download_button=True
                    )
                    gr.Image(
                        value='/kaggle/working/shap_bar.png',
                        label="SHAP Özellik Önem Sırası",
                        show_download_button=True
                    )

print("\n Gradio başlatılıyor...\n")
demo.launch(share=True, debug=False)



 Gradio başlatılıyor...

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://0307bdfb2f97159d79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 💾 10. Model Kaydetme

In [46]:
import pickle

artifacts = {
    'model': best_model,
    'model_name': best_model_name,
    'scaler': scaler,
    'feature_cols': feature_cols,
    'feature_names_tr': feature_names_tr,
    'vectorizer': matcher.vectorizer,
    'role_skills_map': role_skills_map,
    'all_unique_skills': all_unique_skills,
    'explainer': explainer
}

with open('candidate_model_v3.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("✅ Model v3 kaydedildi!")
print("""
🎉 v3 TAMAMLANDI:
1. ✅ SHAP waterfall → her aday için bireysel açıklama
2. ✅ SHAP güçlü/zayıf yön metni → insan okuyabilir gerekçe
3. ✅ Model ezberlemiyor → Early Stopping + L1/L2 + CV kontrolü
4. ✅ Tabs arayüz → Değerlendirme / SHAP / Global XAI
5. ✅ Detaylı SHAP tablosu → her özellik için 🟢🔴⚪
""")

✅ Model v3 kaydedildi!

🎉 v3 TAMAMLANDI:
1. ✅ SHAP waterfall → her aday için bireysel açıklama
2. ✅ SHAP güçlü/zayıf yön metni → insan okuyabilir gerekçe
3. ✅ Model ezberlemiyor → Early Stopping + L1/L2 + CV kontrolü
4. ✅ Tabs arayüz → Değerlendirme / SHAP / Global XAI
5. ✅ Detaylı SHAP tablosu → her özellik için 🟢🔴⚪



In [47]:
# Eksik değer kontrolü — tablo olarak
import pandas as pd

missing = pd.DataFrame({
    'Sütun': df.columns,
    'Eksik Sayı': df.isnull().sum().values,
    'Eksik %': (df.isnull().sum().values / len(df) * 100).round(2)
})

print("Veri Seti Boyutu:", df.shape)
print()
display(missing)

Veri Seti Boyutu: (1000, 11)



,Sütun,Eksik Sayı,Eksik %
0,Resume_ID,0,0.0
1,Name,0,0.0
2,Skills,0,0.0
3,Experience (Years),0,0.0
4,Education,0,0.0
5,Certifications,274,27.4
6,Job Role,0,0.0
7,Recruiter Decision,0,0.0
8,Salary Expectation ($),0,0.0
9,Projects Count,0,0.0


In [48]:
print(f"Veri Seti Boyutu: {df.shape[0]:,} satır x {df.shape[1]} sütun")
print(f"Eksik Değer Sayısı: {df.isnull().sum().sum()}")
display(df.head())

# Dar tablo — sadece eksik değer olan sütunları göster
missing = df.isnull().sum()
missing = missing[missing > 0].to_frame('Eksik Değer Sayısı')
missing['Eksik %'] = (missing['Eksik Değer Sayısı'] / len(df) * 100).round(1).astype(str) + '%'
display(missing)

Veri Seti Boyutu: 1,000 satır x 11 sütun
Eksik Değer Sayısı: 274


,Resume_ID,Name,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Salary Expectation ($),Projects Count,AI Score (0-100)
0,1,Ashley Ali,"TensorFlow, NLP, Pytorch",10,B.Sc,NaN,AI Researcher,Hire,104895,8,100
1,2,Wesley Roman,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,Hire,113002,1,100
2,3,Corey Sanchez,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,Hire,71766,7,70
3,4,Elizabeth Carney,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,Hire,46848,0,95
4,5,Julie Hill,"SQL, React, Java",4,PhD,NaN,Software Engineer,Hire,87441,9,100


,Eksik Değer Sayısı,Eksik %
Certifications,274,27.4%
